Sample code to calculate the freesupply value for geohash 'dr5rus'. df1 - df6 contain 6 month data split into one month each.

In [ ]:
import pandas as pd

In [ ]:
data1 = '/pathtodata/1.csv'
data2 = '/pathtodata/2.csv'
data3 = '/pathtodata/3.csv'
data4 = '/pathtodata/4.csv'
data5 = '/pathtodata/5.csv'
data6 = '/pathtodata/6.csv'

df1 = pd.read_csv(data1) # month 1
df2 = pd.read_csv(data2) # month 2
df3 = pd.read_csv(data3) # month 3
df4 = pd.read_csv(data4) # month 4
df5 = pd.read_csv(data5) # month 5
df6 = pd.read_csv(data6) # month 6

In [ ]:
#Adding column headers to data

column_names = ["Timestamp", "external_ID", "Vehicle_ID", "GeoHash", "Available"]

# List of your DataFrames
dataframes = [df1, df2, df3, df4, df5, df6]

# Assign column names to each DataFrame
for df in dataframes:
    df.columns = column_names

# Display the first few rows of each DataFrame to verify the column names
for i, df in enumerate(dataframes, start=1):
    print(f"DataFrame {i} head:")
    print(df.head())

In [ ]:
# Create and save a per minute freesupply value dataframe
combined_per_minute_supply = pd.DataFrame()

for df in dataframes:
    # Ensure the Timestamp column is in datetime format
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])

    # Round timestamps to the nearest minute
    df['RoundedTimestamp'] = df['Timestamp'].dt.round('T')

    # Remove duplicates to ensure unique driver ID for each minute
    df_unique = df.drop_duplicates(subset=['RoundedTimestamp', 'external_ID'])

    # Group by the rounded timestamp and count unique drivers
    per_minute_supply = df_unique.groupby('RoundedTimestamp')['external_ID'].nunique().reset_index()

    # Rename the columns
    per_minute_supply.columns = ['Timestamp', 'UniqueDrivers']

    # Append the result to the combined DataFrame
    combined_per_minute_supply = pd.concat([combined_per_minute_supply, per_minute_supply], ignore_index=True)

# If there are overlapping timestamps, aggregate them
combined_per_minute_supply = combined_per_minute_supply.groupby('Timestamp')['UniqueDrivers'].sum().reset_index()

# Save the combined DataFrame to a CSV file
combined_per_minute_supply.to_csv('/pathtosave/combined_per_minute_supply.csv', index=False)

# Display the combined result
print("Combined per minute free supply:")
print(combined_per_minute_supply.head())